# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library. The dataset captures survey data on mental health indicators among residents of Kilifi County, Kenya and follows the AI-ready data standards with rich metadata via a Croissant schema.

### Dataset Source
The dataset is described and accessed via its Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install -q mlcroissant matplotlib seaborn

## 1. Data Loading
Load Croissant metadata and data records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access and display useful metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets and their fields.

Below, we enumerate the available Record Sets in the dataset by their `@id` field and display their human-readable `name` and field details, using only the `@id` for internal referencing as per FAIR principles.

In [ ]:
# List all record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    print(f"    Name: {rs.get('name', 'N/A')}")
    print(f"    Description: {rs.get('description', 'N/A')}")
    # List fields
    print("    Fields:")
    for f in rs.get('field', []):
        print(f"      - {f['@id']}: {f.get('name', '')} ({f.get('dataType', '')})")
    print()

### Previewing a Record
Below, we show the first row from one of the main record sets (replace the `<record_set_id>` value as needed; we use the main survey table).

In [ ]:
# Preview a record from the main record set
main_record_set_id = '@kilifi-mental-health-survey-table' # Use the actual @id from previous listing
for rec in dataset.records(record_set=main_record_set_id):
    print(rec)
    break

## 3. Data Extraction
Let's load all data into pandas DataFrames for further exploration. 

*All dataset entities are referenced by their Croissant `@id` fields.*

In [ ]:
# Replace with the list of record set @ids found in the overview
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in all_record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(recs)
    dataframes[rs_id] = df
    print(f"Loaded record set {rs_id} with shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}\n")

## 4. Exploratory Data Analysis (EDA)
Now, we will conduct basic data processing steps using the DataFrame for the main survey record set. All field (column) names are referenced by their Croissant `@id` for traceability.

Let's select two fields for demonstration:
- Numeric field: `@gad7_score` (GAD-7 scale anxiety total score)
- Grouping field: `@village` (participants' village name)

**Note:** Please use the actual `@id` fields by checking column names printed above.

In [ ]:
# Replace with actual field @id as shown in previous output
record_set_id = '@kilifi-mental-health-survey-table'
df = dataframes[record_set_id]

numeric_field = '@gad7_score'  # GAD-7 total score field @id
group_field = '@village'      # Village field @id

# Clean: Remove missing or invalid values for EDA
df_numeric = df[df[numeric_field].notnull()].copy()

# Filtering: Only rows with valid, high GAD-7 scores
threshold = 9
filtered_df = df_numeric[df_numeric[numeric_field] > threshold].copy()
print(f"Found {filtered_df.shape[0]} records with {numeric_field} > {threshold}.")

# Normalize GAD-7 scores
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by village and compute mean GAD-7 score (if village is present)
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} by {group_field} (top 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of anxiety scores (GAD-7) and how they vary by village.

In [ ]:
# Distribution plot for GAD-7 scores
plt.figure(figsize=(8, 4))
sns.histplot(df_numeric[numeric_field], kde=True, bins=15, color='skyblue')
plt.title('Distribution of GAD-7 (Anxiety) Total Scores')
plt.xlabel('GAD-7 Score')
plt.ylabel('Count')
plt.show()

# Boxplot by village (top 10 villages by count)
if group_field in df_numeric.columns:
    top_villages = df_numeric[group_field].value_counts().nlargest(10).index
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df_numeric[df_numeric[group_field].isin(top_villages)])
    plt.title('GAD-7 Scores by Village (Top 10)')
    plt.xlabel('Village')
    plt.ylabel('GAD-7 Total Score')
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook introduced the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, showing how to access and process its data using Croissant and `mlcroissant`.

Key points:
- Dataset and fields were referenced throughout by their Croissant `@id` to ensure traceability.
- We loaded the main record set, identified numeric psychological indicator fields, and demonstrated filtering, normalization, and group summarization (by village).
- Visualizations revealed both the overall distribution of GAD-7 scores and their spread among the most represented villages.

This dataset is suitable for research, health analytics, and community-level reporting. For your own analysis, further explore its additional fields – such as PHQ-9, demographic attributes, and other recorded mental health indicators.